In [ ]:
!pip install -q unsloth[colab-new]

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("../data/preprocessed/metrics_eval.csv")

with open("../data/preprocessed/system_prompt.txt", encoding="utf-8") as f:
    system_prompt = f.read()

print(f"Samples: {len(df)}")
df.head()

In [ ]:
def format_chat(row):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


df["text"] = df.apply(format_chat, axis=1)
dataset = Dataset.from_pandas(df[["text"]])

print(f"Example length (chars): {len(dataset[0]['text'])}")
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
    ),
)

In [ ]:
trainer.train()

## Проверка

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = df["prompt"].iloc[0]

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(inputs, max_new_tokens=1024, temperature=0.3, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Сохранение

Сохраняем LoRA-адаптер и (опционально) merged-модель в GGUF для Ollama.

In [ ]:
model.save_pretrained("koyash-qwen2.5-7b-lora")
tokenizer.save_pretrained("koyash-qwen2.5-7b-lora")

In [ ]:
model.save_pretrained_gguf(
    "koyash-qwen2.5-7b-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

### Загрузка GGUF в Ollama

После скачивания GGUF-файла на локальную машину:

```bash
# Создать Modelfile
echo 'FROM ./koyash-qwen2.5-7b-gguf/unsloth.Q4_K_M.gguf' > Modelfile

# Импортировать в Ollama
ollama create koyash-qwen2.5-7b-lora -f Modelfile

# Проверить
ollama run koyash-qwen2.5-7b-lora
```